# Subscriber Churn Prediction

**Objective**: Build an end-to-end ML pipeline to predict subscriber churn using Snowflake's native ML capabilities.

This notebook demonstrates Snowflake's **complete ML platform**:

| Feature | Description |
|---------|-------------|
| **Snowpark ML** | Distributed model training with scikit-learn APIs |
| **Feature Store** | Managed feature views with automatic refresh |
| **Experiment Tracking** | Compare hyperparameters and metrics across runs |
| **Model Registry** | Version and manage models |
| **Model Serving** | Real-time inference via SPCS |
| **ML Observability** | Monitor model drift and performance |
| **ML Explainability** | SHAP values for predictions |
| **ML Jobs** | Scheduled batch scoring |

## Setup

Initialize the Snowpark session on **Container Runtime** for full Python ecosystem access.

First, install required packages using pip:

In [ ]:
!pip install -r requirements.txt

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# ========================================
# TRAINING & DEPLOYMENT FLAGS
# ========================================
RUN_TRAINING = True
NUM_RUNS = 1
REBUILD_SERVICE = True  # Set True to rebuild SPCS service with new model

# ========================================
# VERSION CONFIGURATION
# Increment these when you want new versions
# ========================================
MODEL_VERSION = 'V32'           # Model version name (increment: V5, V6, ...)
FEATURE_VIEW_VERSION = 'V1'    # Feature view version (rarely changes)
DATASET_VERSION = 'V32'         # Training dataset version (match MODEL_VERSION)

# ========================================
# ENVIRONMENT CONFIGURATION
# ========================================
# For AWS demo account:
DATABASE = 'CHURN_PREDICTION_DEMO'
SCHEMA_RAW = 'RAW'
SCHEMA_FEATURES = 'FEATURES'
SCHEMA_ML = 'ML'
WAREHOUSE = 'COMPUTE_WH'
COMPUTE_POOL = 'CHURN_MODEL_POOL'

# For alternative environments, update these values:
# DATABASE = 'YOUR_DATABASE'
# SCHEMA_RAW = 'YOUR_RAW_SCHEMA'
# SCHEMA_FEATURES = 'YOUR_FEATURES_SCHEMA'
# SCHEMA_ML = 'YOUR_ML_SCHEMA'
# WAREHOUSE = 'YOUR_WAREHOUSE'
# COMPUTE_POOL = 'YOUR_COMPUTE_POOL'
# ========================================

session = get_active_session()
session.use_database(DATABASE)
session.use_schema(SCHEMA_RAW)

st.success(f"Connected to: {session.get_current_database()}.{session.get_current_schema()}")
st.info(f"Config: WAREHOUSE={WAREHOUSE}, COMPUTE_POOL={COMPUTE_POOL}")
st.info(f"Versions: MODEL={MODEL_VERSION}, FEATURE_VIEWS={FEATURE_VIEW_VERSION}, DATASET={DATASET_VERSION}")
if not RUN_TRAINING:
    st.warning("RUN_TRAINING = False. Set to True to train new models.")
else:
    st.info(f"Training configured for {NUM_RUNS} experiment run(s)")
if REBUILD_SERVICE:
    st.warning("REBUILD_SERVICE = True. SPCS service will be rebuilt with new model.")

## 1. Data Exploration

Understand the data structure and churn patterns before building features.

In [ ]:
tables = ['SUBSCRIBERS', 'ENGAGEMENT_EVENTS', 'PAYMENTS', 'EMAIL_INTERACTIONS', 
          'SUBSCRIPTION_HISTORY', 'CUSTOMER_SUPPORT', 'PROMOTIONS', 'ARTICLES']

table_counts = []
for table in tables:
    count = session.table(table).count()
    table_counts.append({'Table': table, 'Row Count': count})

counts_df = pd.DataFrame(table_counts)

fig = px.bar(
    counts_df, x='Table', y='Row Count',
    title='Data Volume by Table',
    color='Row Count',
    color_continuous_scale='Blues',
    text='Row Count'
)
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
st.plotly_chart(fig, use_container_width=True)

In [ ]:
subscribers_df = session.table('SUBSCRIBERS')
history_df = session.table('SUBSCRIPTION_HISTORY')

churn_stats = history_df.filter(F.col('EVENT_TYPE') == 'cancel').group_by().count()
total_subs = subscribers_df.count()
churned = churn_stats.collect()[0][0]
active = total_subs - churned

col1, col2, col3 = st.columns(3)
col1.metric("Total Subscribers", f"{total_subs:,}")
col2.metric("Churned", f"{churned:,}", delta=f"-{churned/total_subs*100:.1f}%", delta_color="inverse")
col3.metric("Active", f"{active:,}", delta=f"{active/total_subs*100:.1f}%")

churn_pie = pd.DataFrame({
    'Status': ['Active', 'Churned'],
    'Count': [active, churned]
})

fig = px.pie(
    churn_pie, values='Count', names='Status',
    title='Overall Churn Distribution',
    color='Status',
    color_discrete_map={'Active': '#2ecc71', 'Churned': '#e74c3c'},
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
st.plotly_chart(fig, use_container_width=True)

In [ ]:
churned_subs = history_df.filter(F.col('EVENT_TYPE') == 'cancel').select('SUBSCRIBER_ID')

churn_by_tier = subscribers_df.join(
    churned_subs, 
    subscribers_df['SUBSCRIBER_ID'] == churned_subs['SUBSCRIBER_ID'],
    'left'
).select(
    subscribers_df['SUBSCRIPTION_TIER'],
    F.when(churned_subs['SUBSCRIBER_ID'].is_not_null(), 1).otherwise(0).alias('CHURNED')
).group_by('SUBSCRIPTION_TIER').agg(
    F.count('*').alias('TOTAL'),
    F.sum('CHURNED').alias('CHURNED'),
    (F.sum('CHURNED') / F.count('*') * 100).alias('CHURN_RATE_PCT')
).order_by('SUBSCRIPTION_TIER')

tier_df = churn_by_tier.to_pandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Subscribers by Tier', 'Churn Rate by Tier'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

fig.add_trace(
    go.Bar(x=tier_df['SUBSCRIPTION_TIER'], y=tier_df['TOTAL'], name='Total', marker_color='#3498db'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=tier_df['SUBSCRIPTION_TIER'], y=tier_df['CHURNED'], name='Churned', marker_color='#e74c3c'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=tier_df['SUBSCRIPTION_TIER'], 
        y=tier_df['CHURN_RATE_PCT'],
        name='Churn Rate %',
        marker_color=tier_df['CHURN_RATE_PCT'],
        marker_colorscale='RdYlGn_r',
        text=tier_df['CHURN_RATE_PCT'].round(1),
        texttemplate='%{text}%',
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_layout(title='Churn Analysis by Subscription Tier', barmode='group', height=400)
st.plotly_chart(fig, use_container_width=True)

st.dataframe(tier_df, use_container_width=True)

## 2. Feature Store

### What is a Feature Store?

A **Feature Store** is a centralized repository for storing, managing, and serving machine learning features. Think of it as a "data warehouse for ML features."

**Key Benefits:**
- **Reusability**: Features defined once, used across multiple models
- **Consistency**: Same feature logic for training and inference (no training-serving skew)
- **Freshness**: Automated pipelines keep features up-to-date
- **Discovery**: Catalog of available features with documentation
- **Point-in-time correctness**: Prevents data leakage by retrieving features as they existed at prediction time

### Snowflake Feature Store Architecture

Snowflake's Feature Store uses **Dynamic Tables** under the hood:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Raw Tables    │────▶│  Feature Views  │────▶│ Training Dataset│
│  (Source Data)  │     │ (Dynamic Tables)│     │  (Point-in-time)│
└─────────────────┘     └─────────────────┘     └─────────────────┘
         │                       │
         │ CDC (Change Data      │ Auto-refresh
         │ Capture)              │ (configurable lag)
         ▼                       ▼
┌─────────────────────────────────────────────────────────────────┐
│                    Incremental Processing                        │
│  Only new/changed rows processed on each refresh                │
└─────────────────────────────────────────────────────────────────┘
```

**Key Components:**
- **Entity**: The subject features describe (e.g., SUBSCRIBER_ID)
- **Feature View**: A logical grouping of related features (e.g., engagement_features)
- **Dataset**: Training data generated by joining spine with feature views

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

session.sql("CREATE SCHEMA IF NOT EXISTS CHURN_PREDICTION_DEMO.FEATURES").collect()

fs = FeatureStore(
    session=session,
    database='CHURN_PREDICTION_DEMO',
    name='FEATURES',
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

st.success("Feature Store initialized: CHURN_PREDICTION_DEMO.FEATURES")

In [ ]:
subscriber_entity = Entity(
    name='SUBSCRIBER',
    join_keys=['SUBSCRIBER_ID'],
    desc='Media Company subscriber entity'
)

try:
    fs.register_entity(subscriber_entity)
    st.info("Entity registered: SUBSCRIBER (join_key: SUBSCRIBER_ID)")
except Exception as e:
    if "already exists" in str(e).lower():
        st.info("Entity SUBSCRIBER already exists - skipping")
        subscriber_entity = fs.get_entity('SUBSCRIBER')
    else:
        raise

In [ ]:
engagement_query = """
SELECT
    SUBSCRIBER_ID,
    COUNT(*) AS TOTAL_ENGAGEMENTS,
    COUNT(DISTINCT ARTICLE_ID) AS UNIQUE_ARTICLES,
    COUNT(DISTINCT SESSION_ID) AS TOTAL_SESSIONS,
    AVG(TIME_SPENT_SECONDS) AS AVG_TIME_SPENT,
    AVG(SCROLL_DEPTH_PCT) AS AVG_SCROLL_DEPTH,
    SUM(CASE WHEN EVENT_TYPE = 'view' THEN 1 ELSE 0 END) AS TOTAL_VIEWS,
    SUM(CASE WHEN EVENT_TYPE = 'share' THEN 1 ELSE 0 END) AS TOTAL_SHARES,
    SUM(CASE WHEN EVENT_TYPE = 'comment' THEN 1 ELSE 0 END) AS TOTAL_COMMENTS,
    COUNT(DISTINCT DEVICE_TYPE) AS DEVICE_DIVERSITY,
    MAX(EVENT_TIMESTAMP) AS LAST_ENGAGEMENT_DATE
FROM CHURN_PREDICTION_DEMO.RAW.ENGAGEMENT_EVENTS
GROUP BY SUBSCRIBER_ID
"""

try:
    engagement_fv = fs.get_feature_view('ENGAGEMENT_FEATURES', FEATURE_VIEW_VERSION)
    st.info("Feature view ENGAGEMENT_FEATURES already exists - skipping")
except:
    engagement_fv = FeatureView(
        name='ENGAGEMENT_FEATURES',
        entities=[subscriber_entity],
        feature_df=session.sql(engagement_query),
        refresh_freq='1 day',
        desc='Subscriber engagement metrics from article interactions'
    )
    engagement_fv = fs.register_feature_view(feature_view=engagement_fv, version=FEATURE_VIEW_VERSION, block=True)
    st.success("Created feature view: ENGAGEMENT_FEATURES")

In [ ]:
payment_query = """
SELECT
    SUBSCRIBER_ID,
    COUNT(*) AS TOTAL_PAYMENTS,
    SUM(CASE WHEN PAYMENT_STATUS = 'failed' THEN 1 ELSE 0 END) AS FAILED_PAYMENTS,
    SUM(CASE WHEN PAYMENT_STATUS = 'success' THEN 1 ELSE 0 END) AS SUCCESSFUL_PAYMENTS,
    AVG(AMOUNT) AS AVG_PAYMENT_AMOUNT,
    SUM(AMOUNT) AS TOTAL_REVENUE,
    SUM(CASE WHEN PAYMENT_STATUS = 'failed' THEN 1 ELSE 0 END)::FLOAT / NULLIF(COUNT(*), 0) AS PAYMENT_FAILURE_RATE
FROM CHURN_PREDICTION_DEMO.RAW.PAYMENTS
GROUP BY SUBSCRIBER_ID
"""

try:
    payment_fv = fs.get_feature_view('PAYMENT_FEATURES', FEATURE_VIEW_VERSION)
    st.info("Feature view PAYMENT_FEATURES already exists - skipping")
except:
    payment_fv = FeatureView(
        name='PAYMENT_FEATURES',
        entities=[subscriber_entity],
        feature_df=session.sql(payment_query),
        refresh_freq='1 day',
        desc='Payment history and failure metrics'
    )
    payment_fv = fs.register_feature_view(feature_view=payment_fv, version=FEATURE_VIEW_VERSION, block=True)
    st.success("Created feature view: PAYMENT_FEATURES")

In [ ]:
email_query = """
SELECT
    SUBSCRIBER_ID,
    SUM(CASE WHEN EVENT_TYPE = 'sent' THEN 1 ELSE 0 END) AS EMAILS_SENT,
    SUM(CASE WHEN EVENT_TYPE = 'opened' THEN 1 ELSE 0 END) AS EMAILS_OPENED,
    SUM(CASE WHEN EVENT_TYPE = 'clicked' THEN 1 ELSE 0 END) AS EMAILS_CLICKED,
    SUM(CASE WHEN EVENT_TYPE = 'unsubscribed' THEN 1 ELSE 0 END) AS EMAIL_UNSUBSCRIBES,
    CASE WHEN SUM(CASE WHEN EVENT_TYPE = 'sent' THEN 1 ELSE 0 END) > 0
         THEN SUM(CASE WHEN EVENT_TYPE = 'opened' THEN 1 ELSE 0 END)::FLOAT / SUM(CASE WHEN EVENT_TYPE = 'sent' THEN 1 ELSE 0 END)
         ELSE 0 END AS EMAIL_OPEN_RATE,
    CASE WHEN SUM(CASE WHEN EVENT_TYPE = 'opened' THEN 1 ELSE 0 END) > 0
         THEN SUM(CASE WHEN EVENT_TYPE = 'clicked' THEN 1 ELSE 0 END)::FLOAT / SUM(CASE WHEN EVENT_TYPE = 'opened' THEN 1 ELSE 0 END)
         ELSE 0 END AS EMAIL_CLICK_RATE
FROM CHURN_PREDICTION_DEMO.RAW.EMAIL_INTERACTIONS
GROUP BY SUBSCRIBER_ID
"""

try:
    email_fv = fs.get_feature_view('EMAIL_FEATURES', FEATURE_VIEW_VERSION)
    st.info("Feature view EMAIL_FEATURES already exists - skipping")
except:
    email_fv = FeatureView(
        name='EMAIL_FEATURES',
        entities=[subscriber_entity],
        feature_df=session.sql(email_query),
        refresh_freq='1 day',
        desc='Email engagement metrics'
    )
    email_fv = fs.register_feature_view(feature_view=email_fv, version=FEATURE_VIEW_VERSION, block=True)
    st.success("Created feature view: EMAIL_FEATURES")

In [ ]:
support_query = """
SELECT
    SUBSCRIBER_ID,
    COUNT(*) AS TOTAL_TICKETS,
    SUM(CASE WHEN TICKET_TYPE = 'billing' THEN 1 ELSE 0 END) AS BILLING_TICKETS,
    SUM(CASE WHEN TICKET_TYPE = 'cancellation' THEN 1 ELSE 0 END) AS CANCEL_TICKETS,
    SUM(CASE WHEN PRIORITY = 'high' THEN 1 ELSE 0 END) AS HIGH_PRIORITY_TICKETS
FROM CHURN_PREDICTION_DEMO.RAW.CUSTOMER_SUPPORT
GROUP BY SUBSCRIBER_ID
"""

try:
    support_fv = fs.get_feature_view('SUPPORT_FEATURES', FEATURE_VIEW_VERSION)
    st.info("Feature view SUPPORT_FEATURES already exists - skipping")
except:
    support_fv = FeatureView(
        name='SUPPORT_FEATURES',
        entities=[subscriber_entity],
        feature_df=session.sql(support_query),
        refresh_freq='1 day',
        desc='Customer support ticket metrics'
    )
    support_fv = fs.register_feature_view(feature_view=support_fv, version=FEATURE_VIEW_VERSION, block=True)
    st.success("Created feature view: SUPPORT_FEATURES")

In [ ]:
promo_query = """
SELECT
    SUBSCRIBER_ID,
    COUNT(*) AS TOTAL_PROMOS,
    MAX(DISCOUNT_PCT) AS MAX_DISCOUNT_PCT,
    SUM(CASE WHEN PROMO_TYPE = 'trial' THEN 1 ELSE 0 END) AS TRIAL_PROMOS
FROM CHURN_PREDICTION_DEMO.RAW.PROMOTIONS
GROUP BY SUBSCRIBER_ID
"""

try:
    promo_fv = fs.get_feature_view('PROMO_FEATURES', FEATURE_VIEW_VERSION)
    st.info("Feature view PROMO_FEATURES already exists - skipping")
except:
    promo_fv = FeatureView(
        name='PROMO_FEATURES',
        entities=[subscriber_entity],
        feature_df=session.sql(promo_query),
        refresh_freq='1 day',
        desc='Promotion and discount history'
    )
    promo_fv = fs.register_feature_view(feature_view=promo_fv, version=FEATURE_VIEW_VERSION, block=True)
    st.success("Created feature view: PROMO_FEATURES")

In [ ]:
fv_list = fs.list_feature_views().to_pandas()

st.subheader("Registered Feature Views")
feature_views_display = []
for _, row in fv_list.iterrows():
    feature_views_display.append({
        'Name': row['NAME'],
        'Version': row['VERSION'],
        'Status': '✅ Active'
    })

fig = go.Figure(data=[go.Table(
    header=dict(values=['Feature View', 'Version', 'Status'],
                fill_color='#3498db', font=dict(color='white', size=14)),
    cells=dict(values=[[fv['Name'] for fv in feature_views_display],
                       [fv['Version'] for fv in feature_views_display],
                       [fv['Status'] for fv in feature_views_display]],
               fill_color='#ecf0f1', font=dict(size=12)))
])
fig.update_layout(title='Feature Store: Registered Feature Views', height=250)
st.plotly_chart(fig, use_container_width=True)

### Feature View Validation

Verify feature views have adequate coverage before generating training data.

In [ ]:
subscriber_count = session.table('RAW.SUBSCRIBERS').count()
fv_counts = {}
for fv_name in ['ENGAGEMENT', 'PAYMENT', 'EMAIL', 'SUPPORT', 'PROMO']:
    fv_counts[fv_name] = session.sql(f"SELECT COUNT(*) FROM FEATURES.{fv_name}_FEATURES${FEATURE_VIEW_VERSION}").collect()[0][0]

required_fvs = ['ENGAGEMENT', 'PAYMENT']  # EMAIL moved to optional
optional_fvs = ['EMAIL', 'SUPPORT', 'PROMO']

required_coverage = min(fv_counts[fv] for fv in required_fvs)
if required_coverage < subscriber_count * 0.9:
    st.error(f"Required feature views may be stale! Subscribers: {subscriber_count:,}, Min coverage: {required_coverage:,}")
    for name in required_fvs:
        st.write(f"  {name}: {fv_counts[name]:,}")
    raise ValueError("Feature views need refresh - re-run notebook or manually refresh dynamic tables")

st.success(f"Feature view validation passed: {subscriber_count:,} subscribers")
for name, count in fv_counts.items():
    status = "✓ required" if name in required_fvs else "(optional)"
    st.write(f"  {name}: {count:,} {status}")

## 3. Generate Training Dataset from Feature Store

Use the Feature Store to generate a training dataset with point-in-time correctness.

In [ ]:
subs_df = session.table('RAW.SUBSCRIBERS')
history_df = session.table('RAW.SUBSCRIPTION_HISTORY')

subscriber_base = subs_df.select(
    'SUBSCRIBER_ID',
    'SUBSCRIPTION_TIER',
    'BILLING_CYCLE',
    'ACQUISITION_CHANNEL',
    F.datediff('day', F.col('SIGNUP_DATE'), F.current_timestamp()).alias('TENURE_DAYS'),
    (F.year(F.current_timestamp()) - F.col('BIRTH_YEAR')).alias('AGE')
)

churned_ids = history_df.filter(F.col('EVENT_TYPE') == 'cancel').select('SUBSCRIBER_ID').distinct()
churn_label = subs_df.select('SUBSCRIBER_ID').join(
    churned_ids.with_column('CHURNED', F.lit(1)),
    'SUBSCRIBER_ID',
    'left'
).fillna({'CHURNED': 0})

spine_df = subscriber_base.join(churn_label, 'SUBSCRIBER_ID')
st.info(f"Spine DataFrame created: {spine_df.count():,} rows")

st.subheader("Random 60/20/20 Data Split")
st.markdown("""
**Proper ML practice**: Random split ensures consistent class distribution across all sets.

| Split | Percentage | Purpose |
|-------|------------|---------|
| **Train** | 60% | Fit model weights |
| **Validation** | 20% | Tune hyperparameters |
| **Test** | 20% | Final unbiased evaluation |
""")

In [ ]:
st.info("Refreshing feature views to ensure latest data...")
for fv_name in ['ENGAGEMENT', 'PAYMENT', 'EMAIL', 'SUPPORT', 'PROMO']:
    session.sql(f"ALTER DYNAMIC TABLE FEATURES.{fv_name}_FEATURES${FEATURE_VIEW_VERSION} REFRESH").collect()
    st.write(f"  ✓ {fv_name}_FEATURES refreshed")
st.success("All feature views refreshed")

from snowflake.ml import dataset

st.info(f"Generating training dataset version {DATASET_VERSION}...")
training_dataset = fs.generate_dataset(
    name='CHURN_TRAINING_DATA',
    version=DATASET_VERSION,
    spine_df=spine_df,
    features=[
        fs.get_feature_view('ENGAGEMENT_FEATURES', FEATURE_VIEW_VERSION),
        fs.get_feature_view('PAYMENT_FEATURES', FEATURE_VIEW_VERSION),
        fs.get_feature_view('EMAIL_FEATURES', FEATURE_VIEW_VERSION),
        fs.get_feature_view('SUPPORT_FEATURES', FEATURE_VIEW_VERSION),
        fs.get_feature_view('PROMO_FEATURES', FEATURE_VIEW_VERSION)
    ],
    desc='Training dataset for churn prediction model'
)
st.success(f"Created dataset: CHURN_TRAINING_DATA version {DATASET_VERSION}")

feature_df = training_dataset.read.to_snowpark_dataframe()

from snowflake.snowpark.types import IntegerType, FloatType, DoubleType, DecimalType, LongType
numeric_types = (IntegerType, FloatType, DoubleType, DecimalType, LongType)
numeric_cols = [f.name for f in feature_df.schema.fields if isinstance(f.datatype, numeric_types)]
feature_df = feature_df.na.fill(0, subset=numeric_cols)

churn_dist = feature_df.group_by('CHURNED').count().to_pandas()
st.subheader("Churn Label Distribution")
st.dataframe(churn_dist, use_container_width=True)
if churn_dist['CHURNED'].sum() == 0 or len(churn_dist) < 2:
    st.error("WARNING: No churned subscribers in dataset! Check spine_df creation in previous cell.")

col1, col2 = st.columns(2)
col1.metric("Total Features", len(feature_df.columns))
col2.metric("Total Rows", f"{feature_df.count():,}")

st.subheader("Training Dataset Sample")
st.dataframe(feature_df.limit(5).to_pandas(), use_container_width=True)

In [ ]:
feature_sample = feature_df.limit(2000).to_pandas()

numeric_features = ['TENURE_DAYS', 'AGE', 'TOTAL_ENGAGEMENTS', 'PAYMENT_FAILURE_RATE', 'EMAIL_OPEN_RATE', 'TOTAL_TICKETS']
available_features = [f for f in numeric_features if f in feature_sample.columns]

fig = make_subplots(rows=2, cols=3, subplot_titles=available_features)

for i, feat in enumerate(available_features):
    row = i // 3 + 1
    col = i % 3 + 1
    
    churned_vals = feature_sample[feature_sample['CHURNED'] == 1][feat]
    active_vals = feature_sample[feature_sample['CHURNED'] == 0][feat]
    
    fig.add_trace(go.Histogram(x=active_vals, name='Active', marker_color='#2ecc71', opacity=0.7, showlegend=(i==0)), row=row, col=col)
    fig.add_trace(go.Histogram(x=churned_vals, name='Churned', marker_color='#e74c3c', opacity=0.7, showlegend=(i==0)), row=row, col=col)

fig.update_layout(title='Feature Distributions: Churned vs Active', height=500, barmode='overlay')
st.plotly_chart(fig, use_container_width=True)

## 4. Experiment Tracking

### What is Experiment Tracking?

**Experiment Tracking** is the practice of logging and comparing different model configurations during development. It answers questions like:
- "Which hyperparameters produced the best model?"
- "How does model A compare to model B?"
- "Can I reproduce last week's best result?"

### Why It Matters

Without experiment tracking, ML development becomes chaotic:
- Lost configurations ("What settings did I use for that good model?")
- No reproducibility ("Why can't I get the same results?")
- Difficult collaboration ("Which model should we deploy?")

### Snowflake Experiment Tracking

```python
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session)
exp.set_experiment('CHURN_MODEL_EXPERIMENT')

with exp.start_run("xgboost_run_1"):
    # Log hyperparameters
    exp.log_params({'n_estimators': 100, 'max_depth': 6})
    
    # Train model...
    
    # Log metrics
    exp.log_metrics({'accuracy': 0.92, 'f1_score': 0.88})
```

**Key Features:**
- **Automatic logging**: Captures environment, timestamps, user
- **Parameter tracking**: Store hyperparameters for each run
- **Metric comparison**: Compare accuracy, F1, etc. across runs
- **Snowsight integration**: Visual comparison in the UI

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

session.sql("CREATE SCHEMA IF NOT EXISTS CHURN_PREDICTION_DEMO.ML").collect()
session.use_schema('ML')

exp = ExperimentTracking(session=session)
exp.set_experiment('CHURN_MODEL_EXPERIMENT')

st.success("Experiment created: CHURN_MODEL_EXPERIMENT")

## XGBoost: The Algorithm

**XGBoost** (eXtreme Gradient Boosting) is the dominant algorithm for tabular/structured data. It consistently wins Kaggle competitions and is the go-to choice for churn prediction.

**How It Works:**

```
Round 1: Build Tree 1 → Predictions have errors
Round 2: Build Tree 2 to correct Tree 1's errors
Round 3: Build Tree 3 to correct remaining errors
...
Final: Sum predictions from all trees
```

**Key Hyperparameters:**

| Parameter | Description | Impact |
|-----------|-------------|--------|
| `n_estimators` | Number of trees | More = better fit, risk of overfitting |
| `max_depth` | Tree depth | Deeper = captures complex patterns |
| `learning_rate` | Step size | Smaller = more robust, slower training |
| `subsample` | % of data per tree | <1.0 reduces overfitting |

**Why XGBoost for Churn:**
- Handles mixed data types (numeric + categorical)
- Robust to missing values
- Captures non-linear feature interactions
- Provides built-in feature importance
- Fast training on large datasets

**Snowflake Integration:**
```python
from snowflake.ml.modeling.xgboost import XGBClassifier

model = XGBClassifier(
    input_cols=feature_columns,
    label_cols=['CHURNED'],
    output_cols=['PREDICTION'],
    n_estimators=100,
    max_depth=6
)
model.fit(train_df)  # Trains inside Snowflake
```

In [ ]:
from snowflake.ml.modeling.preprocessing import OneHotEncoder, StandardScaler
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import accuracy_score, precision_score, recall_score, f1_score

categorical_cols = ['SUBSCRIPTION_TIER', 'BILLING_CYCLE', 'ACQUISITION_CHANNEL']
numeric_cols = [
    'TENURE_DAYS', 'AGE', 'TOTAL_ENGAGEMENTS', 'UNIQUE_ARTICLES', 'TOTAL_SESSIONS',
    'AVG_TIME_SPENT', 'AVG_SCROLL_DEPTH', 'TOTAL_VIEWS', 'TOTAL_SHARES', 'TOTAL_COMMENTS',
    'DEVICE_DIVERSITY', 'TOTAL_PAYMENTS', 'FAILED_PAYMENTS', 'SUCCESSFUL_PAYMENTS',
    'AVG_PAYMENT_AMOUNT', 'TOTAL_REVENUE', 'PAYMENT_FAILURE_RATE', 'EMAILS_SENT',
    'EMAILS_OPENED', 'EMAILS_CLICKED', 'EMAIL_UNSUBSCRIBES', 'EMAIL_OPEN_RATE',
    'EMAIL_CLICK_RATE', 'TOTAL_TICKETS', 'BILLING_TICKETS', 'CANCEL_TICKETS',
    'HIGH_PRIORITY_TICKETS', 'TOTAL_PROMOS', 'MAX_DISCOUNT_PCT', 'TRIAL_PROMOS'
]
target_col = 'CHURNED'

model_df = feature_df.select(
    ['SUBSCRIBER_ID'] + categorical_cols + numeric_cols + [target_col]
).dropna()

model_df_with_rand = model_df.with_column('RAND_VAL', F.abs(F.hash(F.col('SUBSCRIBER_ID'))) % 100)

train_df = model_df_with_rand.filter(F.col('RAND_VAL') < 60).drop('RAND_VAL', 'SUBSCRIBER_ID')
val_df = model_df_with_rand.filter((F.col('RAND_VAL') >= 60) & (F.col('RAND_VAL') < 80)).drop('RAND_VAL', 'SUBSCRIBER_ID')
test_df = model_df_with_rand.filter(F.col('RAND_VAL') >= 80).drop('RAND_VAL', 'SUBSCRIBER_ID')

test_with_id = model_df_with_rand.filter(F.col('RAND_VAL') >= 80).drop('RAND_VAL')

st.subheader("Random 60/20/20 Split Results")
col1, col2, col3 = st.columns(3)
col1.metric("Train (60%)", f"{train_df.count():,} rows")
col2.metric("Validation (20%)", f"{val_df.count():,} rows")
col3.metric("Test (20%)", f"{test_df.count():,} rows")

st.info("✅ Deterministic split using SUBSCRIBER_ID hash ensures consistent class distribution.")

# === DEBUG: Data Quality Checks ===
st.subheader("🔍 Debug: Data Quality Checks")

# Check split sizes and class distribution
train_count = train_df.count()
val_count = val_df.count()
test_count = test_df.count()
train_churn = train_df.filter(F.col('CHURNED')==1).count()
val_churn = val_df.filter(F.col('CHURNED')==1).count()
st.write(f"**Split sizes:** Train={train_count:,}, Val={val_count:,}, Test={test_count:,}")
st.write(f"**Churn rates:** Train={train_churn/train_count:.2%}, Val={val_churn/val_count:.2%}")

# Check data types
st.write("**Train DataFrame dtypes:**")
st.write(train_df.dtypes)

# Check for NULLs in train
null_counts = {}
for c in train_df.columns:
    null_count = train_df.filter(F.col(c).is_null()).count()
    if null_count > 0:
        null_counts[c] = null_count
if null_counts:
    st.warning(f"NULLs found in train: {null_counts}")
else:
    st.success("No NULLs in train_df")

# Check for unseen categories in val
st.write("**Checking for unseen categories in val...**")
unseen_issues = []
for col in categorical_cols:
    train_vals = set([r[0] for r in train_df.select(col).distinct().collect()])
    val_vals = set([r[0] for r in val_df.select(col).distinct().collect()])
    unseen = val_vals - train_vals
    if unseen:
        unseen_issues.append(f"{col}: {unseen}")
if unseen_issues:
    st.warning(f"Unseen categories in val: {unseen_issues}")
else:
    st.success("No unseen categories in val_df")

# Check categorical values
st.write("**Categorical column values in train:**")
for col in categorical_cols:
    vals = [r[0] for r in train_df.select(col).distinct().collect()]
    st.write(f"  {col}: {vals}")

In [ ]:
best_model = None
best_f1 = -1
best_metrics = {}

if not RUN_TRAINING:
    st.info("Skipping model training (RUN_TRAINING = False). Using existing model from registry.")
else:
    import time
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, message='.*column-vector.*')
    warnings.filterwarnings('ignore', category=UserWarning, message='.*sample input.*')
    from snowflake.ml.modeling.preprocessing import LabelEncoder
    from snowflake.ml.modeling.pipeline import Pipeline

    churn_counts = train_df.group_by(target_col).count().to_pandas()
    neg_count = churn_counts[churn_counts[target_col] == 0]['COUNT'].values[0]
    pos_count = churn_counts[churn_counts[target_col] == 1]['COUNT'].values[0]
    st.info(f"Class balance: {neg_count:,} negative, {pos_count:,} positive ({pos_count/(neg_count+pos_count)*100:.1f}% churn rate)")

    def generate_hyperparameter_configs(num_runs):
        if num_runs == 1:
            return [{'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1}]
        elif num_runs == 2:
            return [
                {'n_estimators': 30, 'max_depth': 4, 'learning_rate': 0.3},
                {'n_estimators': 40, 'max_depth': 5, 'learning_rate': 0.25},
            ]
        elif num_runs == 3:
            return [
                {'n_estimators': 30, 'max_depth': 4, 'learning_rate': 0.3},
                {'n_estimators': 40, 'max_depth': 5, 'learning_rate': 0.25},
                {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 0.2},
            ]
        else:
            configs = []
            n_estimators_range = np.linspace(30, 30 + (num_runs - 1) * 10, num_runs, dtype=int)
            max_depth_range = np.linspace(4, 4 + (num_runs - 1), num_runs, dtype=int)
            learning_rate_range = np.linspace(0.3, 0.15, num_runs)
            for i in range(num_runs):
                configs.append({
                    'n_estimators': int(n_estimators_range[i]),
                    'max_depth': int(max_depth_range[i]),
                    'learning_rate': round(learning_rate_range[i], 3)
                })
            return configs

    hyperparameter_configs = generate_hyperparameter_configs(NUM_RUNS)
    st.write(f"Hyperparameter grid ({len(hyperparameter_configs)} configs):", hyperparameter_configs)

    experiment_results = []

    progress_bar = st.progress(0)
    status_text = st.empty()

    run_timestamp = int(time.time())

    for i, config in enumerate(hyperparameter_configs):
        run_name = f"xgboost_run_{run_timestamp}_{i+1}"
        status_text.text(f"Training {run_name}...")

        with exp.start_run(run_name):
            exp.log_params(config)

            pipeline_steps = []
            for col in categorical_cols:
                pipeline_steps.append(
                    (f"LABEL_ENCODER_{col}", LabelEncoder(input_cols=[col], output_cols=[col]))
                )
            pipeline_steps.append(
                ("XGBOOST", XGBClassifier(
                    input_cols=categorical_cols + numeric_cols,
                    label_cols=[target_col],
                    output_cols=['PREDICTION'],
                    n_jobs=-1,
                    tree_method='hist',
                    **config
                ))
            )
            
            pipeline = Pipeline(steps=pipeline_steps)
            pipeline.fit(train_df)

            val_predictions = pipeline.predict(val_df)
            
            # === DEBUG: Check predictions ===
            null_preds = val_predictions.filter(F.col('PREDICTION').is_null()).count()
            total_preds = val_predictions.count()
            st.write(f"**DEBUG [{run_name}]:** Predictions={total_preds}, NULLs={null_preds}")
            if null_preds > 0:
                st.error(f"WARNING: {null_preds} NULL predictions found!")
                st.write("Sample rows with NULL predictions:")
                st.write(val_predictions.filter(F.col('PREDICTION').is_null()).limit(5).to_pandas())
            
            val_accuracy = accuracy_score(df=val_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])
            val_precision = precision_score(df=val_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])
            val_recall = recall_score(df=val_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])
            val_f1 = f1_score(df=val_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])

            exp.log_metrics({'val_accuracy': val_accuracy, 'val_precision': val_precision, 'val_recall': val_recall, 'val_f1_score': val_f1})

            experiment_results.append({
                'RUN_NAME': run_name,
                'N_ESTIMATORS': config['n_estimators'],
                'MAX_DEPTH': config['max_depth'],
                'LEARNING_RATE': config['learning_rate'],
                'ACCURACY': float(val_accuracy),
                'PRECISION': float(val_precision),
                'RECALL': float(val_recall),
                'F1_SCORE': float(val_f1),
                'RUN_TIMESTAMP': pd.Timestamp.now()
            })

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_model = pipeline
                best_metrics = {'accuracy': val_accuracy, 'precision': val_precision, 'recall': val_recall, 'f1_score': val_f1}

        progress_bar.progress((i + 1) / len(hyperparameter_configs))

    status_text.text("Training complete!")
    st.success(f"Best Validation F1 Score: {best_f1:.4f}")
    
    st.subheader("Final Test Set Evaluation")
    st.markdown("*Test set is evaluated ONCE after hyperparameter tuning is complete.*")
    
    test_predictions = best_model.predict(test_df)
    test_accuracy = accuracy_score(df=test_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])
    test_precision = precision_score(df=test_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])
    test_recall = recall_score(df=test_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])
    test_f1 = f1_score(df=test_predictions, y_true_col_names=[target_col], y_pred_col_names=['PREDICTION'])
    
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Test Accuracy", f"{test_accuracy:.4f}")
    col2.metric("Test Precision", f"{test_precision:.4f}")
    col3.metric("Test Recall", f"{test_recall:.4f}")
    col4.metric("Test F1", f"{test_f1:.4f}")
    
    best_metrics = {'accuracy': test_accuracy, 'precision': test_precision, 'recall': test_recall, 'f1_score': test_f1}
    
    if best_model is None:
        st.error("Training completed but best_model is still None - check for errors above")
    else:
        st.success(f"best_model successfully trained: {type(best_model)}")

    results_df = pd.DataFrame(experiment_results)
    
    session.sql("CREATE TABLE IF NOT EXISTS CHURN_PREDICTION_DEMO.ML.EXPERIMENT_RUNS (RUN_NAME STRING, N_ESTIMATORS INT, MAX_DEPTH INT, LEARNING_RATE FLOAT, ACCURACY FLOAT, PRECISION FLOAT, RECALL FLOAT, F1_SCORE FLOAT, RUN_TIMESTAMP TIMESTAMP)").collect()
    session.create_dataframe(results_df).write.mode('append').save_as_table('CHURN_PREDICTION_DEMO.ML.EXPERIMENT_RUNS')
    st.info(f"Saved {len(results_df)} run(s) to EXPERIMENT_RUNS table")

In [ ]:
# Quick validation: ensure model doesn't predict all 1s or all 0s
if best_model is not None:
    st.subheader("Model Prediction Sanity Check")
    
    # Sample 100 rows from validation set
    sample_df = val_df.sample(n=100)
    sample_predictions = best_model.predict(sample_df)
    
    # Count predictions
    pred_counts = sample_predictions.group_by('PREDICTION').count().to_pandas()
    
    st.write("Prediction distribution on 100 validation samples:")
    for _, row in pred_counts.iterrows():
        st.write(f"  Class {int(row['PREDICTION'])}: {row['COUNT']} predictions")
    
    # Check for degenerate predictions
    if len(pred_counts) == 1:
        only_class = int(pred_counts['PREDICTION'].iloc[0])
        st.error(f"⚠️ MODEL ISSUE: All 100 samples predicted as class {only_class}!")
        st.warning("The model is not learning - check features, hyperparameters, or scale_pos_weight")
    else:
        pred_0 = pred_counts[pred_counts['PREDICTION'] == 0]['COUNT'].values[0] if 0 in pred_counts['PREDICTION'].values else 0
        pred_1 = pred_counts[pred_counts['PREDICTION'] == 1]['COUNT'].values[0] if 1 in pred_counts['PREDICTION'].values else 0
        ratio = pred_1 / (pred_0 + pred_1) * 100
        st.success(f"✅ Model predicts both classes: {pred_0} zeros, {pred_1} ones ({ratio:.1f}% positive)")
else:
    st.warning("No model trained - skipping sanity check")

In [ ]:
st.subheader("Experiment Results Comparison")

try:
    all_results_df = session.table('CHURN_PREDICTION_DEMO.ML.EXPERIMENT_RUNS').to_pandas()
except:
    all_results_df = pd.DataFrame()

if len(all_results_df) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Metrics by Run', 'Hyperparameter Impact on F1'),
        specs=[[{'type': 'bar'}, {'type': 'scatter'}]]
    )

    metrics = ['ACCURACY', 'PRECISION', 'RECALL', 'F1_SCORE']
    colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

    for j, metric in enumerate(metrics):
        fig.add_trace(
            go.Bar(x=all_results_df['RUN_NAME'], y=all_results_df[metric], name=metric.replace('_', ' ').title(), marker_color=colors[j]),
            row=1, col=1
        )

    for idx, row in all_results_df.iterrows():
        fig.add_trace(
            go.Scatter(
                x=[row['MAX_DEPTH']],
                y=[row['F1_SCORE']],
                mode='markers+text',
                marker=dict(size=max(row['N_ESTIMATORS']/3, 10), color=row['F1_SCORE'], colorscale='Viridis', cmin=0, cmax=1),
                text=[row['RUN_NAME']],
                textposition='top center',
                name=row['RUN_NAME'],
                showlegend=False
            ),
            row=1, col=2
        )

    fig.update_layout(height=400, barmode='group')
    fig.update_xaxes(title_text='max_depth', row=1, col=2)
    fig.update_yaxes(title_text='F1 Score', row=1, col=2)
    st.plotly_chart(fig, use_container_width=True)

    st.subheader(f"All Experiment Runs ({len(all_results_df)} total)")
    display_df = all_results_df[['RUN_NAME', 'N_ESTIMATORS', 'MAX_DEPTH', 'LEARNING_RATE', 'ACCURACY', 'PRECISION', 'RECALL', 'F1_SCORE', 'RUN_TIMESTAMP']].sort_values('F1_SCORE', ascending=False)
    st.dataframe(display_df, use_container_width=True)

    best_run = all_results_df.loc[all_results_df['F1_SCORE'].idxmax()]
    st.success(f"Best Model: {best_run['RUN_NAME']} with F1 Score = {best_run['F1_SCORE']:.4f}")
else:
    st.warning("No experiment runs found. Run training first to generate experiment data.")

## 5. Model Registry

### What is a Model Registry?

A **Model Registry** is a centralized repository for managing machine learning models throughout their lifecycle. It provides:

- **Versioning**: Track different model versions (v1, v2, v3...)
- **Metadata**: Store hyperparameters, metrics, training data info
- **Lineage**: Know which data and code produced each model
- **Deployment**: Promote models to production environments
- **Governance**: Access control, audit trails, approval workflows

### Why It Matters

| Without Registry | With Registry |
|------------------|---------------|
| Models saved as random files | Centralized, discoverable models |
| "Which pickle file is production?" | Clear version labels and stages |
| No audit trail | Full lineage and history |
| Manual deployment | One-click deployment to SPCS |

### Snowflake Model Registry

```python
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name='DB', schema_name='ML')

# Log a trained model
model_ref = registry.log_model(
    model=trained_pipeline,
    model_name='CHURN_PREDICTION_MODEL',
    version_name='V1',
    metrics={'accuracy': 0.92, 'f1_score': 0.88},
    sample_input_data=sample_df,
    comment='XGBoost trained on 300k subscribers'
)

# Deploy to SPCS for real-time inference
model_ref.create_service(
    service_name='CHURN_INFERENCE_SERVICE',
    compute_pool='CHURN_MODEL_POOL'
)
```

**Key Features:**
- Models stored securely in Snowflake (no external storage needed)
- Native integration with SPCS for serving
- SQL and Python APIs for inference
- Automatic serialization/deserialization

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name='CHURN_PREDICTION_DEMO', schema_name='ML')

if RUN_TRAINING:
    if best_model is None:
        st.error("best_model is None - training did not complete successfully. Check cell 24.")
        raise ValueError("best_model is None")

    sample_input = train_df.select(categorical_cols + numeric_cols).limit(100)

    st.info(f"Registering model version: {MODEL_VERSION}")
    model_version = registry.log_model(
        model=best_model,
        model_name='CHURN_PREDICTION_MODEL',
        version_name=MODEL_VERSION,
        metrics=best_metrics,
        sample_input_data=sample_input,
        target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
        conda_dependencies=[
            "scikit-learn==1.7.1",
            "xgboost==2.1.4",
            "numpy==1.26.4",
        ],
        options={"relax_version": False},
        comment=f'XGBoost churn model {MODEL_VERSION} with random 60/20/20 split'
    )
    st.success(f"Model registered: {model_version.model_name} v{model_version.version_name}")

    session.sql(f"ALTER MODEL CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTION_MODEL SET DEFAULT_VERSION = '{MODEL_VERSION}'").collect()
    st.info(f"Set {MODEL_VERSION} as default version")

    metrics_display = pd.DataFrame([best_metrics]).T.reset_index()
    metrics_display.columns = ['Metric', 'Value']

    fig = px.bar(
        metrics_display, x='Metric', y='Value',
        title='Registered Model Metrics',
        color='Value', color_continuous_scale='Greens',
        text='Value'
    )
    fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
    fig.update_yaxes(range=[0, 1.1])
    st.plotly_chart(fig, use_container_width=True)
else:
    st.info("Skipping model registration (RUN_TRAINING = False). Loading existing model.")

In [ ]:
st.subheader("Model Registry Contents")
model_list = []
for m in registry.models():
    for v in m.versions():
        model_list.append({'Model': m.name, 'Version': v.version_name})

st.dataframe(pd.DataFrame(model_list), use_container_width=True)

## 6. Model Serving (Real-Time Inference)

### What is Model Inference?

**Model Inference** is the process of using a trained model to make predictions on new data.

**Deployment Patterns:**

| Pattern | Latency | Use Case |
|---------|---------|----------|
| **Batch Inference** | Minutes-hours | Score all subscribers nightly |
| **Real-time Inference** | <1 second | Score on-demand (e.g., cancel page) |
| **Streaming Inference** | Seconds | Score events as they arrive |

### Snowpark Container Services (SPCS)

SPCS is Snowflake's container runtime for deploying models and applications:

```
┌─────────────────────────────────────────────────────────────────┐
│                    Snowflake Account                             │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │                    Compute Pool                           │   │
│  │   ┌─────────────┐  ┌─────────────┐  ┌─────────────┐      │   │
│  │   │  Container  │  │  Container  │  │  Container  │      │   │
│  │   │   (Model)   │  │   (Model)   │  │   (Model)   │      │   │
│  │   └─────────────┘  └─────────────┘  └─────────────┘      │   │
│  └──────────────────────────────────────────────────────────┘   │
│                              ▲                                   │
│                              │ HTTP/gRPC                         │
│  ┌───────────────────────────┴───────────────────────────────┐  │
│  │                    Ingress Endpoint                        │  │
│  │            (Secure, authenticated access)                  │  │
│  └───────────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

**Key Benefits:**
- **Security**: Data never leaves Snowflake
- **Auto-scaling**: Scales based on load (1-N containers)
- **GPU Support**: For deep learning models
- **Pay-per-use**: Only pay for compute when running

In [ ]:
session.sql(f"""
CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL}
    MIN_NODES = 1
    MAX_NODES = 2
    INSTANCE_FAMILY = 'CPU_X64_S'
    AUTO_RESUME = TRUE
""").collect()

st.success(f"Compute pool ready: {COMPUTE_POOL}")

In [ ]:
import logging
import glob

logging.getLogger().setLevel(logging.INFO)

model = registry.get_model('CHURN_PREDICTION_MODEL')
mv = model.version(MODEL_VERSION)

existing_services = session.sql("SHOW SERVICES LIKE 'CHURN_INFERENCE_SERVICE' IN SCHEMA CHURN_PREDICTION_DEMO.ML").collect()

if len(existing_services) > 0 and not REBUILD_SERVICE:
    svc_status = existing_services[0]['status']
    st.info(f"Service CHURN_INFERENCE_SERVICE already exists (status: {svc_status}) - reusing")
    st.warning(f"Set REBUILD_SERVICE = True in setup cell to rebuild with model version {MODEL_VERSION}")
else:
    if len(existing_services) > 0:
        st.info(f"REBUILD_SERVICE = True. Dropping existing service to rebuild with {MODEL_VERSION}...")
        session.sql("DROP SERVICE IF EXISTS CHURN_PREDICTION_DEMO.ML.CHURN_INFERENCE_SERVICE").collect()
    
    st.info(f"Creating inference service with model version {MODEL_VERSION}...")
    try:
        mv.create_service(
            service_name='CHURN_INFERENCE_SERVICE',
            service_compute_pool=COMPUTE_POOL,
            ingress_enabled=True,
            force_rebuild=True
        )
        st.success(f"Model deployed to SPCS: CHURN_INFERENCE_SERVICE (version {mv.version_name})")
    except Exception as e:
        st.error(f"Service creation failed: {e}")
        log_files = glob.glob('/root/.local/state/snowflake-ml/log/model_deploy_*.log')
        if log_files:
            latest_log = max(log_files)
            with open(latest_log) as f:
                st.code(f.read()[-5000:], language='text')
        raise

st.info("HTTP endpoint enabled for external applications")

In [ ]:
st.subheader("Deploy Model to Warehouse")
st.markdown("""
**Note**: Warehouse deployment provides an alternative to SPCS for model inference.
SQL syntax: `SELECT model!predict(...) FROM table`
""")

model_ref = registry.get_model('CHURN_PREDICTION_MODEL')
mv_for_wh = model_ref.version(MODEL_VERSION)

try:
    mv_for_wh.deploy(
        deployment_name='CHURN_WH_DEPLOY',
        target_method='predict',
        permanent=True,
        options={'replace': True}
    )
    st.success(f"Model deployed to warehouse: CHURN_WH_DEPLOY (version {MODEL_VERSION})")
except Exception as e:
    if "already exists" in str(e).lower():
        st.info(f"Warehouse deployment already exists")
    else:
        st.warning(f"Warehouse deployment: {e}")

In [ ]:
st.subheader("Real-time Predictions Demo (on Unseen Data)")
st.markdown("*Sampling from test set (20% holdout) - data model has never seen.*")

sample_with_id = test_with_id.limit(10)

feature_cols_sql = ', '.join(categorical_cols + numeric_cols)
sample_with_id.create_or_replace_temp_view('REALTIME_SAMPLE')

rt_sql = f"""
SELECT 
    SUBSCRIBER_ID,
    CHURNED as ACTUAL,
    CHURN_PREDICTION_DEMO.ML.CHURN_INFERENCE_SERVICE!PREDICT(
        {feature_cols_sql}
    ):PREDICTION::INT as PREDICTION
FROM REALTIME_SAMPLE
"""
rt_df = session.sql(rt_sql).to_pandas()

st.subheader(f"Real-time Predictions via SPCS SQL (Model {mv.version_name})")

rt_df['Status'] = rt_df.apply(
    lambda x: '✅ Correct' if x['ACTUAL'] == x['PREDICTION'] else '❌ Incorrect', axis=1
)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Subscriber ID', 'Actual', 'Predicted', 'Status'],
        fill_color='#3498db',
        font=dict(color='white', size=14)
    ),
    cells=dict(
        values=[rt_df['SUBSCRIBER_ID'], rt_df['ACTUAL'], rt_df['PREDICTION'], rt_df['Status']],
        fill_color=[['#ecf0f1']*len(rt_df), ['#ecf0f1']*len(rt_df), ['#ecf0f1']*len(rt_df),
                    ['#2ecc71' if s == '✅ Correct' else '#e74c3c' for s in rt_df['Status']]],
        font=dict(size=12)
    ))
])
fig.update_layout(height=350)
st.plotly_chart(fig, use_container_width=True)

## 7. Batch Inference

Generate predictions for all subscribers and store in Snowflake.

In [ ]:
model_ref = registry.get_model('CHURN_PREDICTION_MODEL')
MODEL_VERSION = model_ref.default.version_name
st.info(f"Using model version: {MODEL_VERSION}")

st.subheader("Batch Predictions on Test Set (SQL-based SPCS Inference)")
st.markdown("""
**Important**: Predictions are generated on the **test set** (20% holdout).
These subscribers were NOT used for training or validation - simulating real production inference.

**Note**: Using direct SQL calls to SPCS service function for accurate inference.
""")

test_with_id.create_or_replace_temp_view('TEST_DATA_FOR_BATCH')
total_rows = test_with_id.count()
st.info(f"Processing {total_rows:,} test subscribers via SQL SPCS inference")

feature_cols_sql = ', '.join(categorical_cols + numeric_cols)

batch_sql = f"""
SELECT 
    SUBSCRIBER_ID,
    CHURNED as ACTUAL_CHURN,
    CHURN_PREDICTION_DEMO.ML.CHURN_INFERENCE_SERVICE!PREDICT(
        {feature_cols_sql}
    ):PREDICTION::INT as PREDICTED_CHURN,
    CURRENT_TIMESTAMP() as PREDICTION_TIMESTAMP
FROM TEST_DATA_FOR_BATCH
"""

with st.spinner("Generating batch predictions via SPCS SQL..."):
    prediction_df = session.sql(batch_sql)
    prediction_df.write.mode('overwrite').save_as_table('CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTIONS')

pred_count = session.table('CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTIONS').count()
st.success(f"Batch predictions saved: {pred_count:,} rows (test set)")

In [ ]:
confusion_data = session.sql("""
    SELECT 
        ACTUAL_CHURN,
        PREDICTED_CHURN,
        COUNT(*) as COUNT
    FROM CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTIONS
    GROUP BY ACTUAL_CHURN, PREDICTED_CHURN
    ORDER BY ACTUAL_CHURN, PREDICTED_CHURN
""").to_pandas()

tn = confusion_data[(confusion_data['ACTUAL_CHURN']==0) & (confusion_data['PREDICTED_CHURN']==0)]['COUNT'].values
fp = confusion_data[(confusion_data['ACTUAL_CHURN']==0) & (confusion_data['PREDICTED_CHURN']==1)]['COUNT'].values
fn = confusion_data[(confusion_data['ACTUAL_CHURN']==1) & (confusion_data['PREDICTED_CHURN']==0)]['COUNT'].values
tp = confusion_data[(confusion_data['ACTUAL_CHURN']==1) & (confusion_data['PREDICTED_CHURN']==1)]['COUNT'].values

tn = int(tn[0]) if len(tn) > 0 else 0
fp = int(fp[0]) if len(fp) > 0 else 0
fn = int(fn[0]) if len(fn) > 0 else 0
tp = int(tp[0]) if len(tp) > 0 else 0

confusion_matrix = np.array([[tn, fp], [fn, tp]])

fig = px.imshow(
    confusion_matrix,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=['Active (0)', 'Churned (1)'],
    y=['Active (0)', 'Churned (1)'],
    title='Confusion Matrix',
    color_continuous_scale='Blues',
    text_auto=True
)
fig.update_layout(height=400)
st.plotly_chart(fig, use_container_width=True)

col1, col2, col3, col4 = st.columns(4)
col1.metric("True Negatives", f"{tn:,}")
col2.metric("False Positives", f"{fp:,}")
col3.metric("False Negatives", f"{fn:,}")
col4.metric("True Positives", f"{tp:,}")

## 8. ML Observability

### What is ML Observability?

**ML Observability** is the practice of monitoring deployed models to detect performance degradation before it impacts business outcomes.

**Types of Drift:**

| Type | Description | Example |
|------|-------------|---------|
| **Data Drift** | Input feature distributions change | Avg. tenure increases as subscriber base matures |
| **Concept Drift** | Relationship between features and target changes | Economic recession changes churn drivers |
| **Prediction Drift** | Model output distribution shifts | Suddenly predicting 50% churn vs. historical 25% |

### Why It Matters

Models trained on historical data assume the future looks like the past. When this assumption breaks:
- **Silent failures**: Model keeps predicting, but predictions are wrong
- **Delayed detection**: Business impact before anyone notices
- **Expensive retraining**: Catch drift early = smaller retraining dataset needed

### Snowflake ML Observability

```python
# Create a model monitor
session.sql('''
CREATE MODEL MONITOR CHURN_MODEL_MONITOR WITH
    MODEL = CHURN_PREDICTION_MODEL
    VERSION = 'V1'
    FUNCTION = 'PREDICT'
    SOURCE = MODEL_MONITORING_SOURCE
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
''')
```

**Monitored Metrics:**
- **Accuracy decay**: Performance drops over time
- **PSI (Population Stability Index)**: Feature distribution shift
- **Prediction volume**: Unusual spikes or drops
- **Latency**: Inference time trends

**Alerting**: Configure thresholds to trigger notifications when metrics exceed bounds.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE CHURN_PREDICTION_DEMO.ML.MODEL_MONITORING_SOURCE AS
SELECT 
    SUBSCRIBER_ID::VARCHAR AS ID,
    PREDICTION_TIMESTAMP::TIMESTAMP_NTZ AS TIMESTAMP,
    PREDICTED_CHURN::NUMBER AS PREDICTION,
    ACTUAL_CHURN::NUMBER AS ACTUAL
FROM CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTIONS
""").collect()

st.success("Monitoring source table created")

In [ ]:
try:
    try:
        session.sql(f"""
        ALTER MODEL CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTION_MODEL 
        MODIFY VERSION {MODEL_VERSION}
        SET TASK = 'TABULAR_BINARY_CLASSIFICATION'
        """).collect()
    except Exception as e:
        if "already set" not in str(e).lower():
            raise

    session.sql(f"""
    CREATE OR REPLACE MODEL MONITOR CHURN_PREDICTION_DEMO.ML.CHURN_MODEL_MONITOR WITH
        MODEL = CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTION_MODEL
        VERSION = '{MODEL_VERSION}'
        FUNCTION = 'PREDICT'
        SOURCE = CHURN_PREDICTION_DEMO.ML.MODEL_MONITORING_SOURCE
        WAREHOUSE = {session.get_current_warehouse()}
        REFRESH_INTERVAL = '1 day'
        AGGREGATION_WINDOW = '1 day'
        TIMESTAMP_COLUMN = TIMESTAMP
        ID_COLUMNS = ('ID')
        PREDICTION_CLASS_COLUMNS = ('PREDICTION')
        ACTUAL_CLASS_COLUMNS = ('ACTUAL')
    """).collect()

    st.success(f"Model Monitor created: CHURN_MODEL_MONITOR (tracking version {MODEL_VERSION})")
except Exception as e:
    err_str = str(e).lower()
    if "unsupported feature" in err_str or "quality_monitor" in err_str:
        st.warning("Model Monitor feature not enabled in this account - skipping")
    else:
        st.error(f"Monitor creation failed: {e}")

monitor_info = [
    {'Capability': 'Performance Tracking', 'Description': 'Accuracy, Precision, Recall, F1 over time'},
    {'Capability': 'Drift Detection', 'Description': 'PSI, KL Divergence on features'},
    {'Capability': 'Volume Monitoring', 'Description': 'Prediction counts and distribution'},
    {'Capability': 'Alerting', 'Description': 'Threshold-based notifications'}
]

fig = go.Figure(data=[go.Table(
    header=dict(values=['Monitoring Capability', 'Description'], fill_color='#9b59b6', font=dict(color='white', size=14)),
    cells=dict(values=[[m['Capability'] for m in monitor_info], [m['Description'] for m in monitor_info]],
               fill_color='#ecf0f1', font=dict(size=12)))
])
fig.update_layout(title='ML Observability Features', height=220)
st.plotly_chart(fig, use_container_width=True)

In [ ]:
try:
    monitor_desc = session.sql("DESC MODEL MONITOR CHURN_PREDICTION_DEMO.ML.CHURN_MODEL_MONITOR").to_pandas()
    st.subheader("Monitor Configuration")
    st.dataframe(monitor_desc, use_container_width=True)
except Exception as e:
    if "unsupported feature" in str(e).lower():
        st.warning("Model Monitor DESC not available in Container Runtime - view monitor in Snowsight UI")
    else:
        st.error(f"Could not describe monitor: {e}")

## 9. ML Explainability (SHAP Values)

### What is Model Explainability?

**Model Explainability** answers the question: *"Why did the model make this prediction?"*

This is crucial for:
- **Trust**: Stakeholders need to understand predictions before acting
- **Debugging**: Identify when the model is using wrong patterns
- **Compliance**: Regulations (GDPR, CCPA) may require explanations
- **Improvement**: Know which features to engineer or collect more data for

### SHAP Values

**SHAP** (SHapley Additive exPlanations) is the gold standard for model interpretation:

```
Prediction = Baseline + SHAP(feature_1) + SHAP(feature_2) + ... + SHAP(feature_n)
```

**Example:**
```
Base churn probability: 25%

+ Tenure < 30 days:      +15% (new subscribers churn more)
+ Payment failure rate:  +20% (billing issues = high churn)
+ Email open rate > 50%: -10% (engaged = less likely to churn)
─────────────────────────────
Final prediction:         50% churn probability
```

### Interpreting SHAP Plots

**1. Feature Importance (Bar Chart)**
- Shows which features have the most impact *overall*
- Longer bar = more influential feature

**2. Dependence Plots (Scatter)**
- X-axis: Feature value
- Y-axis: SHAP value (impact on prediction)
- Reveals non-linear relationships and thresholds

**3. Waterfall Plots**
- Shows how each feature contributes to a *single* prediction
- Useful for explaining individual decisions

### Snowflake Integration

```python
import shap

# Get the underlying XGBoost model
xgb_model = pipeline.to_sklearn().steps[-1][1]

# Compute SHAP values
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X)

# Save to Snowflake for dashboard use
importance_df.write.save_as_table('ML.FEATURE_IMPORTANCE')
```

In [ ]:
if RUN_TRAINING:
    import shap
    from sklearn.preprocessing import LabelEncoder as SklearnLabelEncoder

    sample_pd = val_df.limit(500).to_pandas()
    sample_pd.columns = [c.strip('"').upper() for c in sample_pd.columns]
    
    sklearn_encoders = {}
    sample_encoded = sample_pd.copy()
    for col in categorical_cols:
        le = SklearnLabelEncoder()
        sample_encoded[col] = le.fit_transform(sample_pd[col].astype(str))
        sklearn_encoders[col] = le
    
    X_explain = sample_encoded[categorical_cols + numeric_cols]
    
    if best_model is None:
        st.info(f"Loading model {MODEL_VERSION} from registry for SHAP analysis...")
        loaded_mv = registry.get_model('CHURN_PREDICTION_MODEL').version(MODEL_VERSION)
        sklearn_pipeline = loaded_mv.load().to_sklearn()
    else:
        sklearn_pipeline = best_model.to_sklearn()
    
    xgb_model = sklearn_pipeline.steps[-1][1]

    with st.spinner("Computing SHAP values..."):
        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(X_explain)

    st.success("SHAP values computed for model explainability")
else:
    st.info("Skipping SHAP computation (RUN_TRAINING = False)")

In [ ]:
if RUN_TRAINING:
    feature_importance = pd.DataFrame({
        'Feature': X_explain.columns,
        'SHAP Importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('SHAP Importance', ascending=True).tail(15)

    fig = px.bar(
        feature_importance, x='SHAP Importance', y='Feature',
        orientation='h',
        title='Top 15 Churn Drivers (SHAP Feature Importance)',
        color='SHAP Importance',
        color_continuous_scale='Reds'
    )
    fig.update_layout(height=500, showlegend=False)
    st.plotly_chart(fig, use_container_width=True)
else:
    st.info("Skipping feature importance plot (RUN_TRAINING = False)")

In [ ]:
if RUN_TRAINING:
    top_features = feature_importance.tail(6)['Feature'].tolist()
    shap_df = pd.DataFrame(shap_values, columns=X_explain.columns)

    fig = make_subplots(rows=2, cols=3, subplot_titles=top_features)

    for i, feat in enumerate(top_features):
        row = i // 3 + 1
        col = i % 3 + 1
        
        fig.add_trace(
            go.Scatter(
                x=X_explain[feat].values,
                y=shap_df[feat].values,
                mode='markers',
                marker=dict(size=4, color=shap_df[feat].values, colorscale='RdBu', opacity=0.6),
                showlegend=False
            ),
            row=row, col=col
        )
        fig.update_xaxes(title_text='Feature Value', row=row, col=col)
        fig.update_yaxes(title_text='SHAP Value', row=row, col=col)

    fig.update_layout(title='SHAP Dependence Plots: Feature Value vs Impact on Prediction', height=600)
    st.plotly_chart(fig, use_container_width=True)
else:
    st.info("Skipping SHAP dependence plots (RUN_TRAINING = False)")

In [ ]:
if RUN_TRAINING:
    importance_save = feature_importance.sort_values('SHAP Importance', ascending=False)
    shap_save_df = session.create_dataframe(
        importance_save.values.tolist(),
        schema=['FEATURE', 'SHAP_IMPORTANCE']
    )
    shap_save_df.write.mode('overwrite').save_as_table('CHURN_PREDICTION_DEMO.ML.FEATURE_IMPORTANCE')

    st.success("Feature importance saved to CHURN_PREDICTION_DEMO.ML.FEATURE_IMPORTANCE")
else:
    st.info("Skipping feature importance save (RUN_TRAINING = False)")

## 10. ML Jobs (Scheduled Batch Scoring)

Set up **Snowflake ML Jobs** for scheduled batch inference on new subscriber data.

In [ ]:
batch_scoring_code = f'''
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.ml.registry import Registry
from snowflake.ml.feature_store import FeatureStore

MODEL_VERSION = '{MODEL_VERSION}'
FEATURE_VIEW_VERSION = '{FEATURE_VIEW_VERSION}'
WAREHOUSE = '{WAREHOUSE}'

def score_subscribers():
    session = Session.builder.getOrCreate()
    session.use_database('CHURN_PREDICTION_DEMO')
    session.use_schema('ML')
    session.use_warehouse(WAREHOUSE)
    
    fs = FeatureStore(
        session=session,
        database='CHURN_PREDICTION_DEMO',
        name='FEATURES',
        default_warehouse=WAREHOUSE
    )
    
    subs_df = session.table('RAW.SUBSCRIBERS').select(
        'SUBSCRIBER_ID',
        'SUBSCRIPTION_TIER',
        'BILLING_CYCLE', 
        'ACQUISITION_CHANNEL',
        F.datediff('day', F.col('SIGNUP_DATE'), F.current_timestamp()).alias('TENURE_DAYS'),
        (F.year(F.current_timestamp()) - F.col('BIRTH_YEAR')).alias('AGE')
    )
    
    feature_df = fs.retrieve_feature_values(
        spine_df=subs_df,
        features=[
            fs.get_feature_view('ENGAGEMENT_FEATURES', FEATURE_VIEW_VERSION),
            fs.get_feature_view('PAYMENT_FEATURES', FEATURE_VIEW_VERSION),
            fs.get_feature_view('EMAIL_FEATURES', FEATURE_VIEW_VERSION),
            fs.get_feature_view('SUPPORT_FEATURES', FEATURE_VIEW_VERSION),
            fs.get_feature_view('PROMO_FEATURES', FEATURE_VIEW_VERSION)
        ]
    ).fillna(0)
    
    registry = Registry(session=session, database_name='CHURN_PREDICTION_DEMO', schema_name='ML')
    model = registry.get_model('CHURN_PREDICTION_MODEL').version(MODEL_VERSION)
    
    predictions = model.run(
        feature_df, 
        function_name='predict',
        service_name='CHURN_INFERENCE_SERVICE'
    )
    
    output = predictions.select(
        'SUBSCRIBER_ID',
        F.col('PREDICTION').alias('PREDICTED_CHURN'),
        F.current_timestamp().alias('SCORED_AT')
    )
    
    output.write.mode('overwrite').save_as_table('CHURN_PREDICTION_DEMO.ML.DAILY_CHURN_SCORES')
    return f"Scored {{output.count()}} subscribers"

if __name__ == "__main__":
    __return__ = score_subscribers()
'''

with open('/tmp/batch_scoring.py', 'w') as f:
    f.write(batch_scoring_code)

print("Batch scoring script created: /tmp/batch_scoring.py")
print(batch_scoring_code)

In [ ]:
from snowflake.ml.jobs import submit_file, list_jobs

session.sql("CREATE STAGE IF NOT EXISTS CHURN_PREDICTION_DEMO.ML.ML_JOBS_STAGE").collect()

job = submit_file(
    '/tmp/batch_scoring.py',
    compute_pool=COMPUTE_POOL,
    stage_name='CHURN_PREDICTION_DEMO.ML.ML_JOBS_STAGE',
    session=session
)

col1, col2 = st.columns(2)
col1.metric("Job ID", job.id)
col2.metric("Status", job.status)

In [ ]:
st.subheader("Recent ML Jobs")
jobs_df = list_jobs(session=session, limit=5)
st.dataframe(jobs_df, use_container_width=True)

## Summary

Complete ML pipeline using Snowflake's native capabilities.

In [ ]:
st.header("Pipeline Summary")

summary_data = [
    {'Component': 'Feature Engineering', 'Snowflake Feature': 'Feature Store', 'Artifact': f'5 Feature Views ({FEATURE_VIEW_VERSION})'},
    {'Component': 'Model Selection', 'Snowflake Feature': 'Experiment Tracking', 'Artifact': f'{NUM_RUNS} tracked run(s)'},
    {'Component': 'Model Storage', 'Snowflake Feature': 'Model Registry', 'Artifact': f'CHURN_PREDICTION_MODEL {MODEL_VERSION}'},
    {'Component': 'Real-time Inference', 'Snowflake Feature': 'Model Serving (SPCS)', 'Artifact': 'CHURN_INFERENCE_SERVICE'},
    {'Component': 'Batch Inference', 'Snowflake Feature': 'Snowpark ML', 'Artifact': 'ML.CHURN_PREDICTIONS'},
    {'Component': 'Model Monitoring', 'Snowflake Feature': 'ML Observability', 'Artifact': 'CHURN_MODEL_MONITOR'},
    {'Component': 'Interpretability', 'Snowflake Feature': 'ML Explainability', 'Artifact': 'SHAP feature importance'},
    {'Component': 'Automation', 'Snowflake Feature': 'ML Jobs', 'Artifact': 'Scheduled batch scoring'},
]

summary_df = pd.DataFrame(summary_data)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Component', 'Snowflake Feature', 'Artifact'],
        fill_color='#2c3e50',
        font=dict(color='white', size=14),
        align='left'
    ),
    cells=dict(
        values=[summary_df['Component'], summary_df['Snowflake Feature'], summary_df['Artifact']],
        fill_color=[['#ecf0f1', '#d5dbdb']*4],
        font=dict(size=12),
        align='left'
    ))
])
fig.update_layout(title='End-to-End ML Pipeline Components', height=350)
st.plotly_chart(fig, use_container_width=True)

st.subheader("Key Artifacts")
col1, col2 = st.columns(2)
with col1:
    st.info("**Feature Store**: CHURN_PREDICTION_DEMO.FEATURES")
    st.info(f"**Model**: CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTION_MODEL ({MODEL_VERSION})")
    st.info("**Predictions**: CHURN_PREDICTION_DEMO.ML.CHURN_PREDICTIONS")
with col2:
    st.info("**Monitor**: CHURN_PREDICTION_DEMO.ML.CHURN_MODEL_MONITOR")
    st.info("**Service**: CHURN_INFERENCE_SERVICE")
    st.info("**Experiment**: CHURN_MODEL_EXPERIMENT")

st.success("Pipeline complete! View results in Snowsight: AI & ML > Models")